# 7. Result analysis

This notebook reads outputs from `6_modeling.ipynb`, visualizes model performance, and evaluates the two LSTM models by delirium status at `t-1`.

Main analyses:
- within `t~t+2` model comparison across LR/RF/XGB/LightGBM/MLP/single-output LSTM
- row-level probability comparison across models
- new-onset evaluation among cases without delirium at `t-1`
- recovery evaluation among cases with delirium at `t-1`
- additional multi-horizon LSTM evaluation by horizon


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score


In [ ]:
# Project paths
PROJECT_DIR = Path.cwd().resolve().parent
DATA_SPLIT_DIR = PROJECT_DIR / "processed" / "data_split"
MODELING_OUTPUT_DIR = PROJECT_DIR / "outputs" / "modeling"
WITHIN_OUTPUT_DIR = MODELING_OUTPUT_DIR / "within_t_plus_2"
FIGURE_DIR = MODELING_OUTPUT_DIR / "figures"

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_SPLIT_DIR:", DATA_SPLIT_DIR)
print("MODELING_OUTPUT_DIR:", MODELING_OUTPUT_DIR)
print("WITHIN_OUTPUT_DIR:", WITHIN_OUTPUT_DIR)
print("FIGURE_DIR:", FIGURE_DIR)


In [ ]:
# Analysis settings
THRESHOLD = 0.5
MODEL_PROB_COLUMNS = {
    "LR": "lr_prob",
    "RF": "rf_prob",
    "XGB": "xgb_prob",
    "LightGBM": "lgbm_prob",
    "MLP": "mlp_prob",
    "Single-output LSTM": "lstm_prob",
}
HORIZONS = ["y_t", "y_t_plus_1", "y_t_plus_2"]


## Load modeling outputs

In [ ]:
# Modeling outputs from 4/5/6 notebooks
binned = pd.read_csv(DATA_SPLIT_DIR / "events_12h_binned_with_split.csv")
within_summary = pd.read_csv(WITHIN_OUTPUT_DIR / "within_t_plus_2_test_metrics_summary.csv")
within_predictions = pd.read_csv(WITHIN_OUTPUT_DIR / "within_t_plus_2_test_predictions_all_models.csv")
multi_predictions = pd.read_csv(MODELING_OUTPUT_DIR / "lstm_gpu_test_predictions.csv")
multi_horizon_metrics = pd.read_csv(MODELING_OUTPUT_DIR / "lstm_gpu_test_metrics_by_horizon.csv")
multi_test_metrics = pd.read_csv(MODELING_OUTPUT_DIR / "lstm_gpu_test_metrics.csv")

print("within_predictions", within_predictions.shape)
print("multi_predictions", multi_predictions.shape)
display(within_summary)
display(multi_test_metrics)


In [ ]:
# Attach anchor-bin previous delirium feature from the binned modeling table
anchor_prev_delirium = (
    binned[["stay_id", "bin", "prev_delirium"]]
    .rename(columns={"bin": "anchor_bin", "prev_delirium": "t_minus_1_delirium"})
)

within_predictions = within_predictions.merge(anchor_prev_delirium, on=["stay_id", "anchor_bin"], how="left")
multi_predictions = multi_predictions.merge(anchor_prev_delirium, on=["stay_id", "anchor_bin"], how="left")

within_predictions["t_minus_1_group"] = np.where(
    within_predictions["t_minus_1_delirium"].astype(float) >= 0.5,
    "prior_delirium",
    "no_prior_delirium",
)
multi_predictions["t_minus_1_group"] = np.where(
    multi_predictions["t_minus_1_delirium"].astype(float) >= 0.5,
    "prior_delirium",
    "no_prior_delirium",
)

t_minus_1_summary = pd.DataFrame(
    [
        {
            "dataset": "within_t_plus_2_full_window_test",
            "t_minus_1_group": group,
            "n": len(group_df),
            "within_t_plus_2_positive_rate": group_df["y_within_t_plus_2_true"].mean(),
        }
        for group, group_df in within_predictions.groupby("t_minus_1_group")
    ]
    + [
        {
            "dataset": "multi_horizon_test",
            "t_minus_1_group": group,
            "n": len(group_df),
            "within_t_plus_2_positive_rate": group_df["y_within_t_plus_2_true"].mean(),
        }
        for group, group_df in multi_predictions.groupby("t_minus_1_group")
    ]
)

display(t_minus_1_summary)


## Metric functions

In [ ]:
def binary_metric_dict(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = THRESHOLD) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    if len(y_true) == 0:
        return {
            "n": 0,
            "positive_rate": np.nan,
            "auroc": np.nan,
            "auprc": np.nan,
            "sensitivity": np.nan,
            "specificity": np.nan,
            "ppv": np.nan,
            "npv": np.nan,
            "tp": 0,
            "fp": 0,
            "tn": 0,
            "fn": 0,
        }

    if len(np.unique(y_true)) == 2:
        auroc = roc_auc_score(y_true, y_prob)
        auprc = average_precision_score(y_true, y_prob)
    else:
        auroc = np.nan
        auprc = np.nan

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "n": int(len(y_true)),
        "positive_rate": float(y_true.mean()),
        "auroc": auroc,
        "auprc": auprc,
        "sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "ppv": tp / (tp + fp) if (tp + fp) else np.nan,
        "npv": tn / (tn + fn) if (tn + fn) else np.nan,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }


def add_metric_row(rows: list[dict], analysis: str, model: str, group: str, target: str, y_true, y_prob):
    row = {
        "analysis": analysis,
        "model": model,
        "t_minus_1_group": group,
        "target": target,
        **binary_metric_dict(y_true, y_prob),
    }
    rows.append(row)


## Overall result visualization

In [ ]:
# Within t+2 model-level test metric visualization
plot_metric_cols = ["auprc", "auroc", "sensitivity", "specificity", "ppv", "npv"]
within_plot = within_summary[["model", *plot_metric_cols]].copy()
within_plot = within_plot.sort_values("auprc", ascending=False)

display(within_plot)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(within_plot["model"], within_plot["auprc"], color="tab:blue", alpha=0.85)
ax.set_xlabel("Model")
ax.set_ylabel("Test AUPRC")
ax.set_title("Within t+2 test AUPRC by model")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=35, ha="right")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "within_t_plus_2_test_auprc_by_model.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 4.5))
for model, prob_col in MODEL_PROB_COLUMNS.items():
    ax.hist(within_predictions[prob_col], bins=30, alpha=0.35, density=True, label=model)
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Density")
ax.set_title("Within t+2 predicted probability distribution")
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "within_t_plus_2_probability_distribution_by_model.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Multi-horizon LSTM horizon metric visualization
display(multi_horizon_metrics)

horizon_plot = multi_horizon_metrics.set_index("horizon")[["auprc", "auroc"]]
fig, ax = plt.subplots(figsize=(7, 4))
horizon_plot.plot(kind="bar", ax=ax, width=0.75)
ax.set_xlabel("Horizon")
ax.set_ylabel("Score")
ax.set_title("Multi-horizon LSTM test metrics")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "result_analysis_multi_horizon_lstm_metrics.png", dpi=200, bbox_inches="tight")
plt.show()


## t-1 delirium stratified evaluation

In [ ]:
# Within t+2 all-model stratified evaluation
within_stratified_rows = []
for model, prob_col in MODEL_PROB_COLUMNS.items():
    no_prior = within_predictions[within_predictions["t_minus_1_group"] == "no_prior_delirium"]
    prior = within_predictions[within_predictions["t_minus_1_group"] == "prior_delirium"]

    add_metric_row(
        within_stratified_rows,
        analysis="new_onset_from_no_prior_delirium",
        model=model,
        group="no_prior_delirium",
        target="within_t_plus_2_delirium",
        y_true=no_prior["y_within_t_plus_2_true"],
        y_prob=no_prior[prob_col],
    )
    add_metric_row(
        within_stratified_rows,
        analysis="recovery_from_prior_delirium",
        model=model,
        group="prior_delirium",
        target="no_delirium_within_t_plus_2",
        y_true=1 - prior["y_within_t_plus_2_true"].astype(int),
        y_prob=1 - prior[prob_col],
    )

within_stratified_metrics = pd.DataFrame(within_stratified_rows)
within_stratified_metrics.to_csv(MODELING_OUTPUT_DIR / "within_t_plus_2_t_minus_1_stratified_metrics.csv", index=False)
display(within_stratified_metrics.sort_values(["analysis", "auprc"], ascending=[True, False]))


In [ ]:
# LSTM-only stratified evaluation: single-output LSTM and multi-horizon LSTM
lstm_stratified_rows = []

single_no_prior = within_predictions[within_predictions["t_minus_1_group"] == "no_prior_delirium"]
single_prior = within_predictions[within_predictions["t_minus_1_group"] == "prior_delirium"]
add_metric_row(
    lstm_stratified_rows,
    analysis="new_onset_from_no_prior_delirium",
    model="Single-output LSTM",
    group="no_prior_delirium",
    target="within_t_plus_2_delirium",
    y_true=single_no_prior["y_within_t_plus_2_true"],
    y_prob=single_no_prior["lstm_prob"],
)
add_metric_row(
    lstm_stratified_rows,
    analysis="recovery_from_prior_delirium",
    model="Single-output LSTM",
    group="prior_delirium",
    target="no_delirium_within_t_plus_2",
    y_true=1 - single_prior["y_within_t_plus_2_true"].astype(int),
    y_prob=1 - single_prior["lstm_prob"],
)

multi_no_prior = multi_predictions[multi_predictions["t_minus_1_group"] == "no_prior_delirium"]
multi_prior = multi_predictions[multi_predictions["t_minus_1_group"] == "prior_delirium"]
add_metric_row(
    lstm_stratified_rows,
    analysis="new_onset_from_no_prior_delirium",
    model="Multi-horizon LSTM",
    group="no_prior_delirium",
    target="within_t_plus_2_delirium_from_horizons",
    y_true=multi_no_prior["y_within_t_plus_2_true"],
    y_prob=multi_no_prior["y_within_t_plus_2_prob_from_horizons"],
)
add_metric_row(
    lstm_stratified_rows,
    analysis="recovery_from_prior_delirium",
    model="Multi-horizon LSTM",
    group="prior_delirium",
    target="no_delirium_within_t_plus_2_from_horizons",
    y_true=1 - multi_prior["y_within_t_plus_2_true"].astype(int),
    y_prob=1 - multi_prior["y_within_t_plus_2_prob_from_horizons"],
)

# Immediate next-step evaluation using y_t from the multi-horizon model
multi_no_prior_y_t = multi_no_prior[multi_no_prior["y_t_mask"] == 1]
multi_prior_y_t = multi_prior[multi_prior["y_t_mask"] == 1]
add_metric_row(
    lstm_stratified_rows,
    analysis="next_step_new_onset_from_no_prior_delirium",
    model="Multi-horizon LSTM",
    group="no_prior_delirium",
    target="y_t_delirium",
    y_true=multi_no_prior_y_t["y_t_true"],
    y_prob=multi_no_prior_y_t["y_t_prob"],
)
add_metric_row(
    lstm_stratified_rows,
    analysis="next_step_recovery_from_prior_delirium",
    model="Multi-horizon LSTM",
    group="prior_delirium",
    target="no_y_t_delirium",
    y_true=1 - multi_prior_y_t["y_t_true"].astype(int),
    y_prob=1 - multi_prior_y_t["y_t_prob"],
)

lstm_stratified_metrics = pd.DataFrame(lstm_stratified_rows)
lstm_stratified_metrics.to_csv(MODELING_OUTPUT_DIR / "lstm_t_minus_1_stratified_metrics.csv", index=False)
display(lstm_stratified_metrics)


In [ ]:
# Horizon-specific multi-horizon LSTM stratified evaluation
horizon_rows = []
for horizon in HORIZONS:
    true_col = f"{horizon}_true"
    mask_col = f"{horizon}_mask"
    prob_col = f"{horizon}_prob"

    no_prior = multi_predictions[(multi_predictions["t_minus_1_group"] == "no_prior_delirium") & (multi_predictions[mask_col] == 1)]
    prior = multi_predictions[(multi_predictions["t_minus_1_group"] == "prior_delirium") & (multi_predictions[mask_col] == 1)]

    add_metric_row(
        horizon_rows,
        analysis="horizon_new_onset_from_no_prior_delirium",
        model="Multi-horizon LSTM",
        group="no_prior_delirium",
        target=f"{horizon}_delirium",
        y_true=no_prior[true_col],
        y_prob=no_prior[prob_col],
    )
    add_metric_row(
        horizon_rows,
        analysis="horizon_recovery_from_prior_delirium",
        model="Multi-horizon LSTM",
        group="prior_delirium",
        target=f"no_{horizon}_delirium",
        y_true=1 - prior[true_col].astype(int),
        y_prob=1 - prior[prob_col],
    )

multi_horizon_stratified_metrics = pd.DataFrame(horizon_rows)
multi_horizon_stratified_metrics.to_csv(MODELING_OUTPUT_DIR / "multi_horizon_lstm_t_minus_1_horizon_metrics.csv", index=False)
display(multi_horizon_stratified_metrics)


## Stratified visualizations

In [ ]:
# Within t+2 stratified AUPRC by model
plot_df = within_stratified_metrics.copy()
plot_df["label"] = np.where(
    plot_df["analysis"] == "new_onset_from_no_prior_delirium",
    "New onset | t-1 no delirium",
    "Recovery | t-1 delirium",
)

fig, ax = plt.subplots(figsize=(10, 5))
for idx, label in enumerate(plot_df["label"].unique()):
    subset = plot_df[plot_df["label"] == label].sort_values("model")
    x = np.arange(len(subset)) + (idx - 0.5) * 0.35
    ax.bar(x, subset["auprc"], width=0.35, label=label)
ax.set_xticks(np.arange(len(subset)))
ax.set_xticklabels(subset["model"], rotation=35, ha="right")
ax.set_xlabel("Model")
ax.set_ylabel("AUPRC")
ax.set_title("Within t+2 stratified AUPRC by t-1 delirium status")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "within_t_plus_2_t_minus_1_stratified_auprc.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# LSTM-only stratified AUPRC and AUROC
lstm_plot = lstm_stratified_metrics[lstm_stratified_metrics["analysis"].isin([
    "new_onset_from_no_prior_delirium",
    "recovery_from_prior_delirium",
])].copy()
lstm_plot["label"] = np.where(
    lstm_plot["analysis"] == "new_onset_from_no_prior_delirium",
    "New onset | t-1 no delirium",
    "Recovery | t-1 delirium",
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, metric in zip(axes, ["auprc", "auroc"]):
    pivot = lstm_plot.pivot(index="model", columns="label", values=metric)
    pivot.plot(kind="bar", ax=ax, width=0.75)
    ax.set_xlabel("Model")
    ax.set_ylabel(metric.upper())
    ax.set_title(f"LSTM {metric.upper()} by t-1 group")
    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "lstm_t_minus_1_stratified_auprc_auroc.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Multi-horizon LSTM horizon-specific stratified AUPRC
horizon_plot = multi_horizon_stratified_metrics.copy()
horizon_plot["horizon"] = horizon_plot["target"].str.replace("no_", "", regex=False).str.replace("_delirium", "", regex=False)
horizon_plot["label"] = np.where(
    horizon_plot["analysis"] == "horizon_new_onset_from_no_prior_delirium",
    "New onset | t-1 no delirium",
    "Recovery | t-1 delirium",
)

fig, ax = plt.subplots(figsize=(8, 4.5))
for idx, label in enumerate(horizon_plot["label"].unique()):
    subset = horizon_plot[horizon_plot["label"] == label].set_index("horizon").loc[HORIZONS]
    x = np.arange(len(HORIZONS)) + (idx - 0.5) * 0.35
    ax.bar(x, subset["auprc"], width=0.35, label=label)
ax.set_xticks(np.arange(len(HORIZONS)))
ax.set_xticklabels(HORIZONS)
ax.set_xlabel("Horizon")
ax.set_ylabel("AUPRC")
ax.set_title("Multi-horizon LSTM stratified AUPRC by horizon")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "multi_horizon_lstm_t_minus_1_horizon_auprc.png", dpi=200, bbox_inches="tight")
plt.show()


## Save augmented prediction tables

In [ ]:
# Row-level prediction tables with t-1 delirium status
within_predictions.to_csv(MODELING_OUTPUT_DIR / "within_t_plus_2_test_predictions_all_models_with_t_minus_1.csv", index=False)
multi_predictions.to_csv(MODELING_OUTPUT_DIR / "lstm_gpu_test_predictions_with_t_minus_1.csv", index=False)

print("Saved result analysis tables and figures")
